# 08 - Match index and over-by-over progress export

The dashboard's historical pages (Match Centre, Replays, Par Lab, Score Engine, Win Probability, Data Explorer) need to know **which matches exist
and what happened in each one**. This notebook re-uses the parsing and cleaning from notebooks 01-07 and writes `matches.json`:

* a **match index** for every YAML file (date, teams, venue, toss, winner, margin, player of the match, innings totals, and the reason a match was excluded, if it was);
* **over-by-over progress** (cumulative runs, wickets and boundaries at the end of every over, both innings) for every usable match.

It needs no model. It contains no predictions: those come from the per-match cells at the end of notebooks 03, 04 and 06 (`predictions.json`).

In [1]:
import os, re, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path(os.environ.get("IPL_PROJECT_ROOT", Path.cwd().parent))
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("YAML files  :", len(list(RAW_DATA_DIR.glob("*.yaml"))))

Project root: c:\Users\amitb\Desktop\ME\Projects\Data_Analysis_Practice\cricket-analytics-lab
YAML files  : 1243


## 1. Parse and clean

In [2]:
NOT_A_WICKET = {"retired hurt", "retired not out"}      # not a lost wicket
BOWLER_WICKET_KINDS = {"bowled", "caught", "lbw", "stumped", "caught and bowled", "hit wicket"}   # credited to the bowler


def iter_raw_deliveries(innings_data):
    if "deliveries" in innings_data:                       # layout used by your files
        for entry in innings_data["deliveries"]:
            if isinstance(entry, dict) and entry:
                yield next(iter(entry.values()))
    elif "overs" in innings_data:                          # newer layout
        for over in innings_data["overs"]:
            for d in over.get("deliveries", []):
                yield d
    else:
        raise ValueError("Unknown innings layout: %s" % list(innings_data))


def parse_delivery(d, reg):
    # reg maps player name -> unique id (info.registry.people); falls back to the name itself
    runs = d.get("runs") or {}
    extras = d.get("extras") or {}
    wk = d.get("wickets")
    if wk is None:
        w = d.get("wicket")
        wk = [w] if isinstance(w, dict) else []
    wk = [w for w in wk if w.get("kind") not in NOT_A_WICKET]
    pid = lambda n: reg.get(n, n)
    batter_runs = runs.get("batter", runs.get("batsman", 0)) or 0
    wides, noballs = extras.get("wides", 0) or 0, extras.get("noballs", 0) or 0
    return {
        "total": runs.get("total", 0) or 0, "batter_runs": batter_runs, "wide": wides, "noball": noballs,
        "wickets": len(wk), "out": [pid(w["player_out"]) for w in wk if w.get("player_out")],
        "bowler_wkts": sum(1 for w in wk if w.get("kind") in BOWLER_WICKET_KINDS),
        "bat": pid(d.get("batsman", d.get("batter"))), "bowl": pid(d.get("bowler")),
        "runs_vs_bowler": batter_runs + wides + noballs,          # byes / leg-byes are not charged to the bowler
    }


def summarise_innings(dels):
    # over-end states {k: (runs, wickets, boundaries, dismissed_player_ids)} + per-player batting / bowling lines
    runs = wkts = bnd = legal = bw_wk = 0
    out, states, batters, bowlers = set(), {}, {}, {}
    for d in dels:
        runs += d["total"]; wkts += d["wickets"]; out.update(d["out"]); bw_wk += d["bowler_wkts"]
        if d["batter_runs"] in (4, 6) and d["wide"] == 0:
            bnd += 1
        if d["wide"] == 0:                                        # a ball faced by the batter
            b = batters.setdefault(d["bat"], [0, 0]); b[0] += 1; b[1] += d["batter_runs"]
        bw = bowlers.setdefault(d["bowl"], [0, 0, 0]); bw[1] += d["runs_vs_bowler"]; bw[2] += d["bowler_wkts"]
        if d["wide"] == 0 and d["noball"] == 0:                   # a legal ball
            legal += 1; bw[0] += 1
            if legal % 6 == 0:
                states[legal // 6] = (runs, wkts, bnd, frozenset(out))
    bat_balls = sum(v[0] for v in batters.values())
    return dict(states=states, runs=runs, wkts=wkts, legal=legal, batters=batters, bowlers=bowlers, bowler_wkts=bw_wk,
                bat_rr=(sum(v[1] for v in batters.values()) / bat_balls) if bat_balls else 0.0,       # batter runs per ball faced
                bowl_rr=(sum(v[1] for v in bowlers.values()) / legal) if legal else 0.0)              # runs charged to bowlers per legal ball


def get_innings(match):
    out = []
    for entry in match.get("innings", []):
        if "team" in entry:                                       # newer layout
            if not entry.get("super_over"):
                out.append(entry)
        else:                                                     # layout used by your files: {"1st innings": {...}}
            name, data = next(iter(entry.items()))
            if name in ("1st innings", "2nd innings"):
                out.append(data)
    return out[:2]


def load_all(raw_dir):
    rows, parsed, names, errors = [], {}, {}, []
    for path in sorted(raw_dir.glob("*.yaml")):
        try:
            with path.open("r", encoding="utf-8") as f:
                match = yaml.safe_load(f)
            info = match["info"]
            reg = (info.get("registry") or {}).get("people") or {}
            for n, i in reg.items():
                names[i] = n
            oc = info.get("outcome") or {}
            row = {"match_id": path.stem, "date": pd.to_datetime(str(info["dates"][0])),
                   "venue_raw": info.get("venue"), "scheduled_overs": info.get("overs", 20),
                   "winner": oc.get("winner"), "method": oc.get("method"), "result": oc.get("result"),
                   "toss_winner": (info.get("toss") or {}).get("winner"), "toss_decision": (info.get("toss") or {}).get("decision"),
                   "by_runs": (oc.get("by") or {}).get("runs"), "by_wickets": (oc.get("by") or {}).get("wickets"),
                   "city": info.get("city"), "stage": (info.get("event") or {}).get("stage"),
                   "match_number": (info.get("event") or {}).get("match_number"),
                   "player_of_match": ((info.get("player_of_match") or [None])[0])}
            xi = {team: [reg.get(p, p) for p in plist] for team, plist in (info.get("players") or {}).items()}
            inns = get_innings(match)
            if len(inns) == 2:
                s = [summarise_innings([parse_delivery(d, reg) for d in iter_raw_deliveries(x)]) for x in inns]
                row.update(team1=inns[0].get("team"), team2=inns[1].get("team"),
                           final_score=s[0]["runs"], first_wkts=s[0]["wkts"], first_legal=s[0]["legal"],
                           second_total=s[1]["runs"], second_wkts=s[1]["wkts"], second_legal=s[1]["legal"])
                parsed[path.stem] = dict(date=row["date"], xi=xi, inn=s, team1=row["team1"], team2=row["team2"])
            rows.append(row)
        except Exception as exc:
            errors.append((path.name, repr(exc)))
    return pd.DataFrame(rows), parsed, names, errors


raw_matches, parsed, player_names, load_errors = load_all(RAW_DATA_DIR)
print("Matches parsed :", len(raw_matches), "| with both innings:", len(parsed))
print("Players named  :", len(player_names))
print("Load errors    :", len(load_errors), load_errors[:3])

Matches parsed : 1243 | with both innings: 1237
Players named  : 964
Load errors    : 0 []


In [3]:
def status(r):
    if pd.isna(r.get("team2")):
        return "missing an innings"
    if r["scheduled_overs"] != 20:
        return "not a 20-over match"
    if pd.notna(r["method"]):
        return "rain-adjusted target (D/L or similar)"
    if not (r["first_legal"] >= 120 or r["first_wkts"] >= 10):
        return "first innings curtailed"
    if r["result"] in ("tie", "no result") or pd.isna(r["winner"]) or r["second_total"] == r["final_score"]:
        return "tie / no result"
    if (r["second_total"] > r["final_score"]) != (r["winner"] == r["team2"]):
        return "label mismatch (scores vs outcome)"
    return "ok"


raw_matches["status"] = raw_matches.apply(status, axis=1)
print(raw_matches["status"].value_counts().to_string())

matches = raw_matches[raw_matches["status"] == "ok"].copy()
matches["chase_won"] = (matches["second_total"] > matches["final_score"]).astype(int)
matches["target"] = matches["final_score"] + 1
matches["season"] = matches["date"].dt.year

VENUE_ALIASES = {
    "feroz shah kotla": "Arun Jaitley Stadium",
    "sardar patel stadium": "Narendra Modi Stadium",
    "sheikh zayed stadium": "Zayed Cricket Stadium",
    "punjab cricket association stadium": "Punjab Cricket Association IS Bindra Stadium",
    "punjab cricket association is bindra stadium": "Punjab Cricket Association IS Bindra Stadium",
    "subrata roy sahara stadium": "Maharashtra Cricket Association Stadium",
}

def standardize_venue(v):
    if not isinstance(v, str):
        return "Unknown"
    key = re.sub(r"\s+", " ", v.split(",")[0].replace(".", " ")).strip().lower()
    return VENUE_ALIASES.get(key, re.sub(r"\s+", " ", v.split(",")[0].replace(".", " ")).strip())

matches["venue"] = matches["venue_raw"].map(standardize_venue)
print("\nUsable matches:", len(matches), "| venue labels: raw %d -> standardised %d" % (
    matches["venue_raw"].nunique(), matches["venue"].nunique()))

status
ok                                       1183
rain-adjusted target (D/L or similar)      23
tie / no result                            18
first innings curtailed                    13
missing an innings                          6

Usable matches: 1183 | venue labels: raw 60 -> standardised 36


## 2. Export `matches.json`

In [4]:
# ===== Match index + over-by-over progress for ALL matches (notebook 08 export) ===========
import json, datetime

DASH_DIR = Path(os.environ.get("IPL_DASHBOARD_DATA", PROJECT_ROOT / "dashboard" / "public" / "data"))
DASH_DIR.mkdir(parents=True, exist_ok=True)
ERA_NAMES = {"Delhi Daredevils": "Delhi Capitals", "Kings XI Punjab": "Punjab Kings",
             "Royal Challengers Bangalore": "Royal Challengers Bengaluru", "Rising Pune Supergiants": "Rising Pune Supergiant"}
_nm = lambda x: ERA_NAMES.get(x, x) if isinstance(x, str) else None
_val = lambda x: None if x is None or (isinstance(x, float) and np.isnan(x)) else (x.item() if hasattr(x, "item") else x)

idx = raw_matches.copy()
idx["season"] = idx["date"].dt.year
idx["venue"] = idx["venue_raw"].map(standardize_venue)
ok_ids = set(matches["match_id"])
mi = matches.set_index("match_id")

COLS = ["id", "date", "season", "team1", "team2", "venue", "city", "stage", "match_number", "toss_winner", "toss_decision", "winner",
        "result", "method", "by_runs", "by_wickets", "player_of_match", "first_runs", "first_wkts", "second_runs", "second_wkts",
        "target", "chase_won", "status"]
rows = []
for r in idx.sort_values(["date", "match_id"]).itertuples():
    g = lambda c: _val(getattr(r, c, None))
    rows.append([r.match_id, str(r.date)[:10], int(r.season), _nm(g("team1")), _nm(g("team2")), r.venue, g("city"), g("stage"),
                 g("match_number"), _nm(g("toss_winner")), g("toss_decision"), _nm(g("winner")), g("result"), g("method"),
                 g("by_runs"), g("by_wickets"), g("player_of_match"), g("final_score"), g("first_wkts"), g("second_total"),
                 g("second_wkts"), int(mi.at[r.match_id, "target"]) if r.match_id in ok_ids else None,
                 int(mi.at[r.match_id, "chase_won"]) if r.match_id in ok_ids else None, r.status])

progress = {}
for mid in parsed:                      # every match with two parsed innings, not only the ones the win-probability models use
    e = {}
    for i, inn in enumerate(parsed[mid]["inn"], start=1):
        st = inn["states"]
        for j, key in enumerate(("r", "w", "b")):
            e[key + str(i)] = [int(st[k][j]) if k in st else None for k in range(1, 21)]
    progress[mid] = e

payload = {"cols": COLS, "rows": rows, "progress": progress,
           "teams": sorted({_nm(t) for t in set(idx["team1"].dropna()) | set(idx["team2"].dropna())}),
           "seasons": sorted(int(s) for s in idx["season"].unique()), "files": int(len(idx)), "usable": int(len(ok_ids)), "with_progress": int(len(parsed)),
           "status_counts": {str(k): int(v) for k, v in idx["status"].value_counts().items()}}
meta = {"notebook": "matches", "generated": datetime.datetime.now().isoformat(timespec="seconds"), "matches": int(len(ok_ids)),
        "first_season": int(idx["season"].min()), "last_season": int(idx["season"].max())}
path = DASH_DIR / "matches.json"
path.write_text(json.dumps({"meta": meta, **payload}, separators=(",", ":"), default=_val), encoding="utf-8")
print("wrote %s (%.0f KB): %d match rows, %d with over-by-over progress" % (path, path.stat().st_size / 1024, len(rows), len(progress)))

wrote c:\Users\amitb\Desktop\ME\Projects\Data_Analysis_Practice\cricket-analytics-lab\dashboard\public\data\matches.json (742 KB): 1243 match rows, 1237 with over-by-over progress
